# 06 - Lakeflow Designer: Preparación Visual de Datos

## Objetivo
Crear una transformación **sin código** con el canvas visual de **Lakeflow Designer**.

## Prerequisitos
- Catálogo con datos (después de `00_setup` y Lab 04)
- Permiso `CAN USE` en compute Serverless

## Duración: ~20 minutos


In [0]:
%run "./00 - Setup/00_variables"


In [0]:
# Datos de entrada para Designer
display(spark.sql(f"SELECT * FROM {catalog_name}.{schema_silver}.transacciones LIMIT 5"))


## Paso 1 — Crear Visual Data Prep

1. En el menú lateral, clic en **New**
2. Seleccione **Visual data prep** (Lakeflow Designer)
3. Nombre: `BNCR Resumen Canales - <usuario>`

## Paso 2 — Conectar fuente de datos

1. En el canvas, agregue un nodo **Read**
2. Seleccione como fuente:
   - **Catálogo:** su catálogo
   - **Esquema:** su `schema_silver` (ej. `rico_silver`)
   - **Tabla:** `transacciones`
3. Clic en **Preview** para ver los datos

## Paso 3 — Transformar (drag & drop)

Agregue operadores en este orden:

| # | Operador | Configuración |
|---|----------|---------------|
| 1 | **Filter** | `monto > 0` y `moneda = 'CRC'` |
| 2 | **Group by** | Agrupar por `canal`, `fecha_transaccion` |
| 3 | **Aggregate** | `COUNT(*)` → `total_txn`, `SUM(monto)` → `monto_total` |
| 4 | **Sort** | Por `monto_total` descendente |

> También puede describir la transformación en **lenguaje natural** en el panel de Genie y Designer generará los pasos.

### Prompt sugerido para Genie en Designer (copiar y pegar)

```
Filtra las transacciones donde monto > 0 y moneda = 'CRC'.
Luego agrupa por canal y fecha_transaccion.
Calcula el conteo total como total_txn y la suma de monto como monto_total.
Ordena por monto_total de mayor a menor.
```

## Paso 4 — Escribir resultado

1. Agregue nodo **Write**
2. Destino Unity Catalog:
   - Catálogo: su catálogo
   - Esquema: su `schema_gold` (ej. `rico_gold`)
   - Tabla: `designer_resumen_canales`
3. Modo: **Overwrite** o **Append**
4. **Preview** final → confirmar

## Paso 5 — Guardar y ejecutar

1. **Save** — se guarda como archivo `.designer.ipynb`
2. Clic **Run** para ejecutar la transformación
3. Opcional: **Schedule** para programar como Job


In [0]:
# Verificar tabla creada por Designer
try:
    display(spark.sql(f"""
      SELECT canal, fecha_transaccion, total_txn, monto_total
      FROM {catalog_name}.{schema_gold}.designer_resumen_canales
      ORDER BY monto_total DESC
      LIMIT 10
    """))
    print("Lakeflow Designer OK")
except Exception as e:
    print("Complete los pasos del Designer primero.")
    print(f"Detalle: {e}")


## Paso 6 — Agregar a un Job (opcional)

1. **Jobs & Pipelines → Create → Job**
2. Agregar tarea tipo **Visual data prep**
3. Seleccionar su archivo `.designer.ipynb`
4. Programar o ejecutar manualmente

## Comparación rápida

| Enfoque | Cuándo usar |
|---------|-------------|
| **SQL manual (Lab 04)** | Control total, producción |
| **Genie Code (Lab 05)** | Crear/editar pipelines con IA |
| **Lakeflow Designer (Lab 06)** | Usuarios de negocio, exploración visual |

## Recursos
- [Lakeflow Designer](https://docs.databricks.com/aws/en/designer/)
- [Crear visual data prep](https://docs.databricks.com/aws/en/designer/build-transformation)


## Paso 7 — Crear un Dashboard con Genie Code

Abra **Genie Code** en cualquier notebook y use este prompt para generar un dashboard AI/BI a partir de las tablas gold:

```
Crea un dashboard AI/BI llamado "BNCR Resumen Operativo" con 4 pestañas:

1. **Resumen General** — KPIs: total de transacciones, monto total CRC,
   clientes únicos, monto promedio. Fuente: [catálogo].[usuario]_gold.resumen_diario_sucursal.

2. **Por Sucursal** — Gráfico de barras horizontal con las 10 sucursales
   con mayor monto total. Tabla detalle debajo con fecha, sucursal, provincia,
   transacciones y monto. Fuente: [catálogo].[usuario]_gold.resumen_diario_sucursal.

3. **Por Canal** — Gráfico de líneas mostrando la tendencia diaria de monto
   por canal (app_movil, sucursal, atm, web, sinpe). Gráfico de dona con
   distribución porcentual. Fuente: [catálogo].[usuario]_gold.resumen_producto_canal.

4. **Clientes** — Tabla con métricas por segmento: total clientes,
   saldo promedio, transacciones promedio. Fuente: [catálogo].[usuario]_gold.metricas_clientes.

Usa filtros de fecha y sucursal en las pestañas 1 y 2.
```

> Reemplace `[catálogo]` y `[usuario]` con sus valores (ej. `classic_stable_paco_catalog` y `rico`).

## Limpieza del Workshop

> **⚠️ Ejecute la celda siguiente SOLO al finalizar el taller.** Elimina todos los recursos creados durante el workshop: esquemas, tablas, volumen, job, query y pipeline.

In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

# ---- 1. Eliminar esquemas (CASCADE borra todas las tablas y volúmenes dentro) ----
for schema in [schema_bronze, schema_silver, schema_gold, schema_raw]:
    try:
        spark.sql(f"DROP SCHEMA IF EXISTS {catalog_name}.{schema} CASCADE")
        print(f"✅ Esquema eliminado: {catalog_name}.{schema}")
    except Exception as e:
        print(f"⚠️  {catalog_name}.{schema}: {e}")

# ---- 2. Eliminar el Job creado ----
try:
    jobs = w.jobs.list(name=f"BNCR Workshop Pipeline - {user_suffix}")
    for j in jobs:
        w.jobs.delete(j.job_id)
        print(f"✅ Job eliminado: {j.job_id} ({j.settings.name})")
except Exception as e:
    print(f"⚠️  Job: {e}")

# ---- 3. Eliminar la query SQL guardada ----
try:
    queries = w.queries.list()
    for q in queries:
        if q.display_name and f"BNCR Gold Resumen - {user_suffix}" in q.display_name:
            w.queries.delete(q.id)
            print(f"✅ Query eliminada: {q.id} ({q.display_name})")
except Exception as e:
    print(f"⚠️  Query: {e}")

# ---- 4. Eliminar el pipeline ----
pipeline_id = "147799ee-abfb-4f5f-ab5f-8af4fc3ad7fb"  # <-- ajuste si su pipeline es diferente
try:
    w.pipelines.delete(pipeline_id)
    print(f"✅ Pipeline eliminado: {pipeline_id}")
except Exception as e:
    print(f"⚠️  Pipeline: {e}")

# ---- 5. Eliminar Visual Data Prep (Designer) si existe ----
try:
    import re
    designer_pattern = f"BNCR Resumen Canales - {user_suffix}"
    items = w.workspace.list(f"/Users/rico.martinez@databricks.com/Latam_resources_spanish/Data_Engineering")
    for item in items:
        if item.path and ".designer" in item.path.lower():
            w.workspace.delete(item.path)
            print(f"✅ Designer eliminado: {item.path}")
except Exception as e:
    print(f"⚠️  Designer: {e}")

# ---- 6. Eliminar Dashboard AI/BI si existe ----
try:
    from databricks.sdk.service.dashboards import LifecycleState
    dashboards = w.lakeview.list()
    for d in dashboards:
        if d.display_name and f"BNCR" in d.display_name and d.lifecycle_state != LifecycleState.TRASHED:
            w.lakeview.trash(d.dashboard_id)
            print(f"✅ Dashboard eliminado: {d.dashboard_id} ({d.display_name})")
except Exception as e:
    print(f"⚠️  Dashboard: {e}")

print("\n🧹 Limpieza completa")